# fact_channel_snapshot_poc (periodic snapshot)

Grain: one channel × one read (`channel_id`, `snapshot_ts`). Counts semi-additive; `*_gained` = vs the previous read (additive, NULL on the first read).

**Incremental:** reads in the window **+** reads of channels whose `channel_key` is still −1. LAG over every read of those channels, window filter after it.

Run by `facts_poc` (task `gold_facts_poc`), after `gold_dims_scd1`. Keys come from the `*_scd1` dims (same `xxhash64` keys as the live dims); no match → −1. Test table: the live fact and the agg tables are unchanged.

In [ ]:
%run ./_scd

# Set up environment

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow

silver = dbutils.widgets.get("silver")
gold = dbutils.widgets.get("gold")
load_type = dbutils.widgets.get("load_type")
from_date, to_date = get_incremental_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))
print(f"load_type = {load_type}, window = [{from_date}, {to_date})")

# Read Source Table

In [ ]:
path = f"{gold}.fact_channel_snapshot_poc"
stg_channels = spark.table(f"{silver}.stg_channels")
dim_channel = spark.table(f"{gold}.dim_channel_scd1").filter("channel_key <> -1")

## Incremental: channels read in the window + channels with an unresolved key

In [ ]:
in_window = (F.col("snapshot_date") >= from_date) & (F.col("snapshot_date") < to_date)

if load_type == "incremental":
    changed_channels = stg_channels.filter(in_window).select("channel_id")
    unresolved_channels = spark.createDataFrame([], "channel_id STRING")
    if spark.catalog.tableExists(path):
        unresolved_channels = spark.table(path).filter("channel_key = -1").select("channel_id").distinct()
    stg_channels = stg_channels.join(changed_channels.union(unresolved_channels).distinct(), "channel_id", "left_semi")

# Transformation

## Previous read of the same channel

In [ ]:
from pyspark.sql.window import Window

previous_read = Window.partitionBy("channel_id").orderBy("snapshot_ts")
reads = (stg_channels
         .withColumn("prev_subscriber_count", F.lag("subscriber_count").over(previous_read))
         .withColumn("prev_view_count", F.lag("view_count").over(previous_read))
         .withColumn("prev_video_count", F.lag("video_count").over(previous_read))
         .withColumn("prev_snapshot_ts", F.lag("snapshot_ts").over(previous_read)))

if load_type == "incremental":
    reads = (reads.filter(in_window)
             .unionByName(reads.join(unresolved_channels, "channel_id", "left_semi"))
             .dropDuplicates(["channel_id", "snapshot_ts"]))

## Join dims and select columns

In [ ]:
result_df = (
    reads.alias("read")
    .join(dim_channel.alias("channel"), on=F.col("read.channel_id") == F.col("channel.channel_id"), how="left")
    .select(
        F.col("read.channel_id").cast("string").alias("channel_id"),
        F.col("read.snapshot_ts").cast("timestamp").alias("snapshot_ts"),
        F.date_format("read.snapshot_date", "yyyyMMdd").cast("int").alias("snapshot_date_key"),
        F.coalesce(F.col("channel.channel_key"), F.lit(-1)).cast("bigint").alias("channel_key"),
        F.col("read.subscriber_count").cast("bigint").alias("subscriber_count"),
        F.col("read.view_count").cast("bigint").alias("view_count"),
        F.col("read.video_count").cast("int").alias("video_count"),
        (F.col("read.subscriber_count") - F.col("read.prev_subscriber_count")).cast("bigint").alias("subscribers_gained"),
        (F.col("read.view_count") - F.col("read.prev_view_count")).cast("bigint").alias("views_gained"),
        (F.col("read.video_count") - F.col("read.prev_video_count")).cast("int").alias("videos_gained"),
        F.col("read.prev_snapshot_ts").isNull().alias("is_first_snapshot"),
    )
)

## Add metadata

In [ ]:
insert_date = datetime.now()
etl_by = "Databricks"
result_df = result_df.withColumn("etl_date", F.lit(insert_date)).withColumn("etl_by", F.lit(etl_by))

## Add checksums

In [ ]:
primary_key = ["channel_id", "snapshot_ts"]
result_df = add_checksums(result_df, primary_key)

## Create table

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {path} (
        channel_id STRING, snapshot_ts TIMESTAMP, snapshot_date_key INT, channel_key BIGINT,
        subscriber_count BIGINT, view_count BIGINT, video_count INT,
        subscribers_gained BIGINT, views_gained BIGINT, videos_gained INT, is_first_snapshot BOOLEAN,
        etl_date TIMESTAMP, etl_by STRING, primary_checksum STRING, record_checksum STRING
    ) USING DELTA
    CLUSTER BY (snapshot_date_key)
    COMMENT 'Periodic snapshot at read grain (poc upsert method): 1 channel x 1 read. Counts semi-additive, gains (vs previous read) additive.'
""")
result_df = result_df.select(*spark.table(path).columns)

# Write to Table

**full**: overwrite (poc `full_refresh`). **incremental**: upsert (poc `upsert`) = MERGE on `primary_checksum`; update only when `record_checksum` changed; insert new rows. Rows are never deleted.

In [ ]:
if load_type == "full":
    full_refresh(result_df, path)
else:
    result_df.createOrReplaceTempView("source")
    spark.sql(f"""
        MERGE INTO {path} AS t
        USING source AS s
        ON t.primary_checksum = s.primary_checksum
        WHEN MATCHED AND t.record_checksum <> s.record_checksum THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

## Check: one row per grain

In [ ]:
n, grain = spark.sql(f"SELECT COUNT(*), COUNT(DISTINCT primary_checksum) FROM {path}").first()
assert n == grain, f"{path}: {n - grain} duplicate rows on (channel_id, snapshot_ts)"
last = spark.sql(f"DESCRIBE HISTORY {path} LIMIT 1").select("operation", "operationMetrics").first()
print(f"✅ {path}: {n} rows ·", last["operation"], dict(last["operationMetrics"] or {}))